In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import folium
print("Folium version:", folium.__version__)

pd.set_option('display.max_columns', 20)

Folium version: 0.20.0


In [13]:
stops = pd.read_csv('data/stops.txt')
stops.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,wheelchair_boarding,intersection_code,reference_place,stop_name_short,stop_place
0,11tbro,NaN,11th Ave & Broadway,32.716268,-117.154649,NaN,NaN,1,NaN,0,NaN,NaN,11tbro,NaN
1,12tS,NaN,12th & Imperial Station,32.706002,-117.153378,NaN,NaN,1,NaN,0,NaN,NaN,12tS,NaN
2,imtS,NaN,12th & Imperial Station Bayside,32.705229,-117.154318,NaN,NaN,1,NaN,0,NaN,NaN,imtS,NaN
3,imp12,NaN,12th & Imperial Transit Center,32.705685,-117.152875,NaN,NaN,1,NaN,0,NaN,NaN,imp12,NaN
4,24tS,NaN,24th Street Station,32.661854,-117.108017,NaN,NaN,1,NaN,0,NaN,NaN,24tS,NaN


In [14]:
routes = pd.read_csv('data/routes.txt')
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_type,route_url,route_color,route_text_color,route_group,route_pattern1,route_pattern2
0,1,MTS,1,Fashion Valley - La Mesa,3,https://www.sdmts.com/schedules-real-time?frag...,000099,FFFFFF,SBMF,1-W,1-E
1,2,MTS,2,Downtown San Diego - 30th & Adams,3,https://www.sdmts.com/schedules-real-time?frag...,000099,FFFFFF,SDTC,2-S1,2-N1
2,3,MTS,3,UCSD Hospital - Euclid Transit Center,3,https://www.sdmts.com/schedules-real-time?frag...,000099,FFFFFF,SBMF,3-N,3-S
3,4,MTS,4,12th & Imperial Trolley - Lomita Village,3,https://www.sdmts.com/schedules-real-time?frag...,000099,FFFFFF,SDTC,4-E,4-W
4,5,MTS,5,Downtown San Diego - Euclid Transit Center,3,https://www.sdmts.com/schedules-real-time?frag...,000099,FFFFFF,SBMF,5-E,5-W


In [15]:
trips = pd.read_csv('data/trips.txt')
stop_times = pd.read_csv('data/stop_times.txt')
calendar = pd.read_csv('data/calendar.txt')
shapes = pd.read_csv('data/shapes.txt')

print(f"trips:       {trips.shape}")
print(f"stop_times:  {stop_times.shape}")
print(f"calendar:    {calendar.shape}")
print(f"shapes:      {shapes.shape}")

print("\n--- Routes already loaded ---")
print(f"routes:      {routes.shape}")  

trips:       (22705, 12)
stop_times:  (671017, 10)
calendar:    (44, 11)
shapes:      (97051, 5)

--- Routes already loaded ---
routes:      (105, 11)


/var/folders/lh/p6zh0fy17c7_3q1lh8vxdhrw0000gn/T/ipykernel_42502/1782879397.py:2: DtypeWarning: Columns (0: stop_headsign) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv('data/stop_times.txt')


In [16]:
# Route types in GTFS are coded numerically:
# 0 = Tram/Trolley, 1 = Subway, 2 = Rail, 3 = Bus, 4 = Ferry
route_type_map = {0: 'Trolley', 1: 'Subway', 2: 'Rail', 3: 'Bus', 4: 'Ferry'}
routes['route_type_name'] = routes['route_type'].map(route_type_map)

print("--- Route Type Breakdown ---")
print(routes['route_type_name'].value_counts())

print("\n--- Sample routes from each type ---")
for rtype in routes['route_type_name'].unique():
    sample = routes[routes['route_type_name']==rtype][['route_short_name','route_long_name']].head(3)
    print(f"\n{rtype}:")
    print(sample.to_string(index=False))

--- Route Type Breakdown ---
route_type_name
Bus        99
Trolley     5
Ferry       1
Name: count, dtype: int64

--- Sample routes from each type ---

Bus:
route_short_name                       route_long_name
               1              Fashion Valley - La Mesa
               2     Downtown San Diego - 30th & Adams
               3 UCSD Hospital - Euclid Transit Center

Trolley:
route_short_name            route_long_name
            Blue           San Ysidro - UTC
          Copper          Santee - El Cajon
           Green El Cajon - 12th & Imperial

Ferry:
route_short_name route_long_name
             NaN  Coronado Ferry


In [17]:
# How many trips does each route run? More trips = more frequent service
trips_per_route = trips.groupby('route_id').size().reset_index(name='trip_count')
trips_per_route = trips_per_route.merge(
    routes[['route_id', 'route_short_name', 'route_long_name', 'route_type_name']],
    on='route_id'
)
trips_per_route = trips_per_route.sort_values('trip_count', ascending=False)

print("--- Top 10 busiest routes (by daily trip count) ---")
print(trips_per_route.head(10).to_string(index=False))

print("\n--- Bottom 10 routes (least frequent) ---")
print(trips_per_route.tail(10).to_string(index=False))

print(f"\nMean trips per route: {trips_per_route['trip_count'].mean():.0f}")
print(f"Median: {trips_per_route['trip_count'].median():.0f}")
print(f"Range: {trips_per_route['trip_count'].min()}-{trips_per_route['trip_count'].max()}")

--- Top 10 busiest routes (by daily trip count) ---
route_id  trip_count route_short_name                           route_long_name route_type_name
     227         750              227             Imperial Beach - Otay Mesa TC             Bus
     215         714              215                            Mid-City Rapid             Bus
       7         693                7   Downtown San Diego - University/College             Bus
     510         627             Blue                          San Ysidro - UTC         Trolley
     929         486              929     12th & Imperial - Iris Transit Center             Bus
       8         477                8                   Old Town - Balboa Av TC             Bus
      12         468               12           12th & Imperial - Skyline Hills             Bus
      13         460               13 Kaiser Hospital  - 24th St Transit Center             Bus
     530         458            Green                El Cajon - 12th & Imperial     

In [18]:
# Force both stop_id columns to the same dtype (string) before merging
stops['stop_id'] = stops['stop_id'].astype(str)
stop_times['stop_id'] = stop_times['stop_id'].astype(str)

# Now safely compute trips per stop
trips_per_stop = stop_times.groupby('stop_id').size().reset_index(name='trip_count')

# Merge with stop coordinates
stops_with_freq = stops.merge(trips_per_stop, on='stop_id', how='left')
stops_with_freq['trip_count'] = stops_with_freq['trip_count'].fillna(0)

print("--- Stop Frequency Distribution ---")
print(stops_with_freq['trip_count'].describe())
print(f"\nStops with 0 trips: {(stops_with_freq['trip_count']==0).sum()}")
print(f"\nTop 10 busiest stops:")
print(stops_with_freq.nlargest(10, 'trip_count')[['stop_name','trip_count']].to_string(index=False))

--- Stop Frequency Distribution ---
count    4373.000000
mean      153.445461
std       132.002369
min         0.000000
25%        57.000000
50%       136.000000
75%       209.000000
max      1360.000000
Name: trip_count, dtype: float64

Stops with 0 trips: 106

Top 10 busiest stops:
                          stop_name  trip_count
                Broadway & Union St      1360.0
                  Broadway & 4th Av      1114.0
           Otay Mesa Transit Center      1082.0
                  Broadway & 1st Av      1056.0
                  Broadway & 3rd Av      1043.0
    Complex Dr & Clairemont Mesa Bl      1034.0
Paradise Valley Rd & Meadowbrook Dr       827.0
 University Av & I-15 Transit Plaza       821.0
                     Santa Fe Depot       793.0
                    India St & C St       791.0


In [21]:
# Center on San Diego
m = folium.Map(location=[32.72, -117.16], zoom_start=11, tiles='cartodbpositron')

def get_color(trip_count):
    if trip_count == 0:        return '#cccccc'
    elif trip_count < 50:      return '#fee08b'
    elif trip_count < 200:     return '#fdae61'
    elif trip_count < 500:     return '#f46d43'
    else:                      return '#a50026'

# Plot every stop, color by frequency
for _, row in stops_with_freq.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=3,
        color=get_color(row['trip_count']),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['stop_name']}<br>Trips/day: {int(row['trip_count'])}"
    ).add_to(m)

# Legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 180px; 
            background: white; border:2px solid grey; z-index:9999; 
            font-size:12px; padding: 10px;">
<b>Trips per day</b><br>
<i style="background:#a50026; width:12px; height:12px; display:inline-block"></i> 500+<br>
<i style="background:#f46d43; width:12px; height:12px; display:inline-block"></i> 200-500<br>
<i style="background:#fdae61; width:12px; height:12px; display:inline-block"></i> 50-200<br>
<i style="background:#fee08b; width:12px; height:12px; display:inline-block"></i> 1-50<br>
<i style="background:#cccccc; width:12px; height:12px; display:inline-block"></i> 0<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save and display
m.save('mts_stops_frequency_map.html')
m